## 1 · Imports

In [ ]:
from anndata import AnnData
from typing import Optional

# Libraries
import anndata as ad
#import matplotlib as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.pyplot import rc_context
from scipy.stats import median_abs_deviation

from functools import partial
import altair as alt
import seaborn as sns
import decoupler as dc
from scipy.sparse import csr_matrix
import os
from pathlib import Path

import mudata as mu
import scanpy as sc
import scirpy as ir
import altair as alt
alt.data_transformers.enable("vegafusion")

import anndata as ad
import numpy as np
import palantir

import numpy as np
import scipy.sparse as sp


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import muon as mu
from muon import MuData

In [ ]:
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz
#from sccoda.util import comp_ana as mod

## 2 · Data Loading

Load mudata

In [ ]:
mdata = mu.read_h5mu("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/002_annotate_mudata_normal.h5mu")

In [ ]:
sc.pl.umap(
    mdata["gex"],
    color=["cell_annotation_05"],
    cmap="Reds",
    vmin=0,
    vmax="p99",
    sort_order=False,
    show=False, frameon=False
)

In [ ]:
sc.pl.umap(
    mdata["gex"],
    color=["sample_id"],
    cmap="Reds",
    vmin=0,
    vmax="p99",
    sort_order=False,
    show=False, frameon=False
)

## 2 · Compositional  


In [ ]:
sample_col = "sample_id"
cluster_col = "cell_annotation_05"   # or "rna_leiden", "gex03"
condition_col = "condition"

In [ ]:
cov_df = (
    mdata["gex"].obs[[sample_col, condition_col]]
    .drop_duplicates()
    .set_index(sample_col)
)

cov_df

In [ ]:
cov_df = (
    mdata["gex"].obs[[sample_col, condition_col]]
    .drop_duplicates()
    .set_index(sample_col)
)

In [ ]:
sccoda_data = dat.from_scanpy(
    mdata["gex"],
    cell_type_identifier=cluster_col,
    sample_identifier=sample_col,
    covariate_df=cov_df
)

sccoda_data

In [ ]:
import pandas as pd

condition_col = "condition"

desired_order = [

    "GF",
    "ctrl",
    "effector"
]

# Set ordered categories
sccoda_data.obs[condition_col] = pd.Categorical(
    sccoda_data.obs[condition_col],
    categories=desired_order,
    ordered=True
)

# Optional: sort rows by category order
sccoda_data.obs = sccoda_data.obs.sort_values(condition_col)

In [ ]:
viz.stacked_barplot(sccoda_data, feature_name="samples")
plt.show()

viz.stacked_barplot(sccoda_data, feature_name=condition_col)
plt.show()

viz.boxplots(
    sccoda_data,
    feature_name=condition_col,
    plot_facets=True,
    y_scale="relative",
    add_dots=True
)
plt.show()

In [ ]:
viz.rel_abundance_dispersion_plot(
    data=sccoda_data,
    abundant_threshold=0.9
)
plt.show()

## Running the model 

In [ ]:
mdata

In [ ]:
sccoda_data.var

In [ ]:
mdata["gex"].obs.sample_id

In [ ]:
import pandas as pd

obs = mdata["gex"].obs.copy()

cell_counts = (
    pd.crosstab(
        obs["sample_id"],
        obs["cell_annotation_05"]
    )
    .reset_index()
)

cell_counts = cell_counts.rename(columns={"sample_id": "Mouse"})

cell_counts

In [ ]:
sample_order = [
    "GF1", "GF2",
    "ctrl1", "ctrl2",
    "effector1", "effector2",

]

cell_counts["Mouse"] = pd.Categorical(
    cell_counts["Mouse"],
    categories=sample_order,
    ordered=True
)

cell_counts = (
    cell_counts
    .sort_values("Mouse")
    .reset_index(drop=True)
)

In [ ]:
cell_counts

In [ ]:
# Convert data to anndata object
data_all = dat.from_pandas(cell_counts, covariate_columns=["Mouse"])

# Extract condition from mouse name and add it as an extra column to the covariates
data_all.obs["Condition"] =  (data_all.obs["Mouse"].str.replace(r"\d+$", "", regex=True))

In [ ]:
data_all.obs

In [ ]:
# Select control and salmonella data
data_cd8 = data_all[data_all.obs["Condition"].isin(["effector", "ctrl"])]
print(data_cd8.obs)

In [ ]:
viz.boxplots(data_cd8, feature_name="Condition",figsize=(5,5))
plt.show()

In [ ]:

from sccoda.util import comp_ana as mod

In [ ]:
model_salm = mod.CompositionalAnalysis(data_cd8, formula="Condition", reference_cell_type="Activated")

In [ ]:
# Run MCMC
sim_results = model_salm.sample_hmc()

In [ ]:
sim_results.summary()

In [ ]:
print(sim_results.credible_effects())

In [ ]:
import pickle as pkl

In [ ]:
# saving
outdir = "/data/scratch/kvalem/projects/2021/honda_microbial_metabolites_2021/20_scripts/40_single-cell-sorted-cd8/40_gex_surface_prot/28012026/results/sccoda/sccoda_model_normal"

sim_results.save(outdir)

# loading
with open(outdir, "rb") as f:
    sim_results_2 = pkl.load(f)

sim_results_2.summary()